In [0]:
# Cell 1 — Load from raw Delta checkpoint
# Always read from Delta, never from raw CSVs again

delta_raw = "/Volumes/workspace/default/hmda_raw/delta/raw_hmda"

df = spark.read.format("delta").load(delta_raw)

print(f"✅ Loaded from Delta checkpoint")
print(f"   Rows    : {df.count():,}")
print(f"   Columns : {len(df.columns)}")

✅ Loaded from Delta checkpoint
   Rows    : 11,730,625
   Columns : 99


In [0]:
# Cell 2 — Type casting using try_cast to safely handle "NA" and "Exempt" strings

from pyspark.sql.functions import col, when, trim, expr

df_typed = df.withColumn(
    "interest_rate",    expr("try_cast(interest_rate as double)")
).withColumn(
    "rate_spread",      expr("try_cast(rate_spread as double)")
).withColumn(
    "property_value",   expr("try_cast(property_value as double)")
).withColumn(
    "income",           expr("try_cast(income as double)")
).withColumn(
    "loan_to_value_ratio", expr("try_cast(loan_to_value_ratio as double)")
).withColumn(
    "total_loan_costs", expr("try_cast(total_loan_costs as double)")
).withColumn(
    "loan_term",        expr("try_cast(loan_term as int)")
).withColumn(
    "intro_rate_period",expr("try_cast(intro_rate_period as int)")
).withColumn(
    "multifamily_affordable_units", expr("try_cast(multifamily_affordable_units as int)")
)

print("✅ Type casting complete")
print("\nUpdated schema for key columns:")
df_typed.select(
    "interest_rate",
    "rate_spread",
    "property_value",
    "income",
    "loan_to_value_ratio"
).printSchema()

✅ Type casting complete

Updated schema for key columns:
root
 |-- interest_rate: double (nullable = true)
 |-- rate_spread: double (nullable = true)
 |-- property_value: double (nullable = true)
 |-- income: double (nullable = true)
 |-- loan_to_value_ratio: double (nullable = true)



In [0]:
# Cell 3 — Filter and clean
# We apply business logic filters to keep only meaningful records
# for our 3 research questions

from pyspark.sql.functions import col, when, lit

df_cleaned = df_typed.filter(
    # Keep only the action_taken codes relevant to our analysis:
    # 1 = Loan originated
    # 2 = Application approved but not accepted
    # 3 = Application denied
    col("action_taken").isin([1, 2, 3])
).filter(
    # Remove records with no race info — can't analyze racial disparity
    col("derived_race").isNotNull()
).filter(
    # Remove records with no ethnicity info
    col("derived_ethnicity").isNotNull()
).filter(
    # Loan amount must be positive and realistic
    # Max $20M filters out data entry errors
    (col("loan_amount") > 0) & (col("loan_amount") <= 20_000_000)
).filter(
    # Income must be positive when present
    col("income").isNull() | (col("income") > 0)
).withColumn(
    # Create a clean binary column: 1 = denied, 0 = approved/originated
    "is_denied",
    when(col("action_taken") == 3, 1).otherwise(0)
).withColumn(
    # Create income brackets for controlling income in analysis
    "income_bracket",
    when(col("income").isNull(),        "Unknown")
    .when(col("income") < 50,          "Low (<50k)")
    .when(col("income") < 100,         "Moderate (50-100k)")
    .when(col("income") < 150,         "Middle (100-150k)")
    .when(col("income") < 250,         "Upper-Middle (150-250k)")
    .otherwise(                         "High (250k+)")
)

print(f"✅ Cleaning complete")
print(f"   Rows before : 11,730,625")
print(f"   Rows after  : {df_cleaned.count():,}")
print(f"   Rows removed: {11_730_625 - df_cleaned.count():,}")

print("\n=== Action taken distribution ===")
df_cleaned.groupBy("action_taken").count().orderBy("action_taken").show()

print("=== Income bracket distribution ===")
df_cleaned.groupBy("income_bracket").count().orderBy("count", ascending=False).show()

✅ Cleaning complete
   Rows before : 11,730,625
   Rows after  : 8,096,218
   Rows removed: 3,634,407

=== Action taken distribution ===
+------------+-------+
|action_taken|  count|
+------------+-------+
|           1|5710473|
|           2| 318107|
|           3|2067638|
+------------+-------+

=== Income bracket distribution ===
+--------------------+-------+
|      income_bracket|  count|
+--------------------+-------+
|  Moderate (50-100k)|2598526|
|   Middle (100-150k)|1807429|
|Upper-Middle (150...|1465990|
|        High (250k+)| 912772|
|          Low (<50k)| 858033|
|             Unknown| 453468|
+--------------------+-------+



In [0]:
# Cell 4 — Outlier detection on numeric columns
# Understanding the distribution before we decide what to keep

from pyspark.sql.functions import percentile_approx, avg, stddev, min, max

print("=== Numeric column statistics ===")
df_cleaned.select(
    "loan_amount",
    "interest_rate",
    "rate_spread",
    "income",
    "property_value",
    "loan_to_value_ratio"
).summary("min", "25%", "50%", "75%", "max", "mean").show(truncate=False)

print("=== Interest rate distribution (originated loans only) ===")
df_cleaned.filter(
    (col("action_taken") == 1) & col("interest_rate").isNotNull()
).select(
    percentile_approx("interest_rate", 0.01).alias("p1"),
    percentile_approx("interest_rate", 0.25).alias("p25"),
    percentile_approx("interest_rate", 0.50).alias("median"),
    percentile_approx("interest_rate", 0.75).alias("p75"),
    percentile_approx("interest_rate", 0.99).alias("p99"),
    avg("interest_rate").alias("mean"),
    stddev("interest_rate").alias("std")
).show(truncate=False)

print("=== Rate spread distribution (originated loans only) ===")
df_cleaned.filter(
    (col("action_taken") == 1) & col("rate_spread").isNotNull()
).select(
    percentile_approx("rate_spread", 0.01).alias("p1"),
    percentile_approx("rate_spread", 0.25).alias("p25"),
    percentile_approx("rate_spread", 0.50).alias("median"),
    percentile_approx("rate_spread", 0.75).alias("p75"),
    percentile_approx("rate_spread", 0.99).alias("p99"),
    avg("rate_spread").alias("mean"),
    stddev("rate_spread").alias("std")
).show(truncate=False)

=== Numeric column statistics ===
+-------+------------------+----------------+------------------+-----------------+-----------------+-------------------+
|summary|loan_amount       |interest_rate   |rate_spread       |income           |property_value   |loan_to_value_ratio|
+-------+------------------+----------------+------------------+-----------------+-----------------+-------------------+
|min    |5000.0            |0.0             |-839.0            |1.0              |5000.0           |0.001              |
|25%    |95000.0           |6.125           |-0.18             |70.0             |255000.0         |60.0               |
|50%    |205000.0          |6.875           |0.344             |107.0            |385000.0         |78.98              |
|75%    |355000.0          |7.74            |1.028             |169.0            |605000.0         |91.525             |
|max    |1.9995E7          |120.0           |3960.0            |1.0E8            |2.147483647E9    |1.09368E10         

In [0]:
# Cell 5 — Remove outliers based on realistic business bounds
# We use p1/p99 as guardrails, not arbitrary numbers
# Every threshold below has a business logic reason

df_no_outliers = df_cleaned.filter(
    # Interest rate: 1% to 20% is realistic for mortgages
    # 0% and 120% are clearly data errors
    col("interest_rate").isNull() | 
    (col("interest_rate").between(1.0, 20.0))
).filter(
    # Rate spread: HMDA docs say realistic range is -10 to +10
    # Anything beyond is a data error
    col("rate_spread").isNull() | 
    (col("rate_spread").between(-10.0, 10.0))
).filter(
    # Property value: $10K to $5M covers 99%+ of residential mortgages
    # 2,147,483,647 is integer overflow sentinel value
    col("property_value").isNull() | 
    (col("property_value").between(10_000, 5_000_000))
).filter(
    # LTV: 0% to 200% (200% allows underwater mortgages)
    # 10 billion % LTV is obviously wrong
    col("loan_to_value_ratio").isNull() | 
    (col("loan_to_value_ratio").between(0, 200))
).filter(
    # Income: $1K to $5M annual (in thousands in HMDA)
    # Keeps virtually all real applicants
    col("income").isNull() | 
    (col("income").between(1, 5_000))
)

before = 8_096_218
after  = df_no_outliers.count()

print(f"✅ Outlier removal complete")
print(f"   Rows before : {before:,}")
print(f"   Rows after  : {after:,}")
print(f"   Rows removed: {before - after:,} ({(before - after)/before*100:.2f}%)")

print("\n=== Clean numeric stats ===")
df_no_outliers.select(
    "loan_amount",
    "interest_rate",
    "rate_spread",
    "income",
    "property_value",
    "loan_to_value_ratio"
).summary("min", "50%", "max", "mean").show(truncate=False)

✅ Outlier removal complete
   Rows before : 8,096,218
   Rows after  : 8,007,205
   Rows removed: 89,013 (1.10%)

=== Clean numeric stats ===
+-------+-----------------+----------------+------------------+------------------+-----------------+-------------------+
|summary|loan_amount      |interest_rate   |rate_spread       |income            |property_value   |loan_to_value_ratio|
+-------+-----------------+----------------+------------------+------------------+-----------------+-------------------+
|min    |5000.0           |1.0             |-10.0             |1.0               |15000.0          |0.001              |
|50%    |205000.0         |6.875           |0.352             |108.0             |385000.0         |78.873             |
|max    |1.9965E7         |18.99           |10.0              |5000.0            |4995000.0        |200.0              |
|mean   |279343.5368521226|7.15974643660057|0.5646469054704762|150.04627735860328|514216.3950339078|73.06866017804538  |
+-------+--

In [0]:
# Cell 6 — Save cleaned data as Delta checkpoint
# This is our "silver" layer — cleaned, typed, filtered
# All downstream analysis reads from here

delta_clean = "/Volumes/workspace/default/hmda_raw/delta/cleaned_hmda"

(
    df_no_outliers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(delta_clean)
)

# Read back to confirm
df_verify = spark.read.format("delta").load(delta_clean)

print(f"✅ Cleaned Delta checkpoint saved")
print(f"   Path    : {delta_clean}")
print(f"   Rows    : {df_verify.count():,}")
print(f"   Columns : {len(df_verify.columns)}")

# Quick final check on our most important columns
print("\n=== Final null check on key analysis columns ===")
from pyspark.sql.functions import col, count, when, round as spark_round

key_cols = ["derived_race", "derived_ethnicity", "action_taken", 
            "is_denied", "loan_amount", "income_bracket", "lei"]

total = df_verify.count()
df_verify.select([
    spark_round(
        count(when(col(c).isNull(), c)) / total * 100, 2
    ).alias(c)
    for c in key_cols
]).show(truncate=False)

✅ Cleaned Delta checkpoint saved
   Path    : /Volumes/workspace/default/hmda_raw/delta/cleaned_hmda
   Rows    : 8,007,205
   Columns : 101

=== Final null check on key analysis columns ===
+------------+-----------------+------------+---------+-----------+--------------+---+
|derived_race|derived_ethnicity|action_taken|is_denied|loan_amount|income_bracket|lei|
+------------+-----------------+------------+---------+-----------+--------------+---+
|0.0         |0.0              |0.0         |0.0      |0.0        |0.0           |0.0|
+------------+-----------------+------------+---------+-----------+--------------+---+



In [0]:
# Cell 7 — Derive neighborhood context columns (FIXED)

from pyspark.sql.functions import col, when

df_enriched = df_no_outliers.withColumn(
    # LMI tract = below 80% of area median income
    "is_lmi_tract",
    when(
        col("tract_to_msa_income_percentage").isNull() | 
        (col("tract_to_msa_income_percentage") == 0), None
    )
    .when(col("tract_to_msa_income_percentage") < 80, 1)
    .otherwise(0)
).withColumn(
    # Tract income category per standard FFIEC definitions
    "tract_income_category",
    when(
        col("tract_to_msa_income_percentage").isNull() | 
        (col("tract_to_msa_income_percentage") == 0), "Unknown"
    )
    .when(col("tract_to_msa_income_percentage") < 50,  "Low Income")
    .when(col("tract_to_msa_income_percentage") < 80,  "Moderate Income")
    .when(col("tract_to_msa_income_percentage") < 120, "Middle Income")
    .otherwise("Upper Income")
).withColumn(
    # Minority race flag for research questions 1 & 2
    "is_minority",
    when(
        col("derived_race").isin([
            "Black or African American",
            "American Indian or Alaska Native",
            "Native Hawaiian or Other Pacific Islander"
        ]) | col("derived_ethnicity").isin(["Hispanic or Latino"]),
        1
    ).otherwise(0)
).withColumn(
    # Simplified race category for grouping
    "race_group",
    when(col("derived_race") == "White",                             "White")
    .when(col("derived_race") == "Black or African American",        "Black")
    .when(col("derived_race") == "Asian",                            "Asian")
    .when(col("derived_ethnicity") == "Hispanic or Latino",          "Hispanic")
    .when(col("derived_race") == "American Indian or Alaska Native", "Native American")
    .otherwise("Other/Unknown")
).withColumn(
    # Minority neighborhood flag — tract is majority minority
    "is_minority_tract",
    when(col("tract_minority_population_percent") >= 50, 1).otherwise(0)
)

print("✅ Enrichment complete")
print(f"   Rows    : {df_enriched.count():,}")
print(f"   Columns : {len(df_enriched.columns)}")

print("\n=== Tract income category distribution ===")
df_enriched.groupBy("tract_income_category") \
    .count().orderBy("count", ascending=False).show()

print("=== Race group distribution ===")
df_enriched.groupBy("race_group") \
    .count().orderBy("count", ascending=False).show()

print("=== Minority flag ===")
df_enriched.groupBy("is_minority") \
    .count().orderBy("is_minority").show()

print("=== Minority tract flag ===")
df_enriched.groupBy("is_minority_tract") \
    .count().orderBy("is_minority_tract").show()

✅ Enrichment complete
   Rows    : 8,007,205
   Columns : 106

=== Tract income category distribution ===
+---------------------+-------+
|tract_income_category|  count|
+---------------------+-------+
|        Middle Income|3396660|
|         Upper Income|2833944|
|      Moderate Income|1424263|
|           Low Income| 230445|
|              Unknown| 121893|
+---------------------+-------+

=== Race group distribution ===
+---------------+-------+
|     race_group|  count|
+---------------+-------+
|          White|4892353|
|  Other/Unknown|1453942|
|          Black| 758768|
|          Asian| 596341|
|       Hispanic| 279532|
|Native American|  26269|
+---------------+-------+

=== Minority flag ===
+-----------+-------+
|is_minority|  count|
+-----------+-------+
|          0|6008143|
|          1|1999062|
+-----------+-------+

=== Minority tract flag ===
+-----------------+-------+
|is_minority_tract|  count|
+-----------------+-------+
|                0|5178376|
|                

In [0]:
# Cell 8 — Save enriched Delta checkpoint
# This is our "gold" layer — ready for analysis

delta_enriched = "/Volumes/workspace/default/hmda_raw/delta/enriched_hmda"

(
    df_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(delta_enriched)
)

df_verify = spark.read.format("delta").load(delta_enriched)

print(f"✅ Enriched Delta checkpoint saved")
print(f"   Path    : {delta_enriched}")
print(f"   Rows    : {df_verify.count():,}")
print(f"   Columns : {len(df_verify.columns)}")

✅ Enriched Delta checkpoint saved
   Path    : /Volumes/workspace/default/hmda_raw/delta/enriched_hmda
   Rows    : 8,007,205
   Columns : 106
